# Ch 32 (검증) — 작동하는 diffusion 입문 + 기본 샘플러

vocab 2048 + 30000 step 학습된 모델에 Ch 32 기본 샘플러(confidence remasking)를 붙여 coherent 생성되는지 확인.

In [ ]:
%pip install -q -U transformers tokenizers datasets accelerate

In [ ]:
import math, time, torch
import torch.nn.functional as F
from datasets import load_dataset

SEED = 42
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = torch.cuda.is_available()
print("torch", torch.__version__, "| device", device, "| fp16", USE_FP16)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
raw_train = load_dataset("roneneldan/TinyStories", split="train[:100000]")
raw_val   = load_dataset("roneneldan/TinyStories", split="validation[:500]")
print(raw_train)
print(raw_val[0]["text"][:160])

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import PreTrainedTokenizerFast

VOCAB = 2048
def corpus_iter(bs=1000):
    for i in range(0, len(raw_train), bs):
        yield raw_train[i:i+bs]["text"]

_tk = Tokenizer(models.BPE(unk_token="[UNK]"))
_tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
_tk.decoder = decoders.ByteLevel()
_trainer = trainers.BpeTrainer(vocab_size=VOCAB, special_tokens=["[PAD]", "[UNK]", "[MASK]"])
_tk.train_from_iterator(corpus_iter(), trainer=_trainer)

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=_tk, pad_token="[PAD]", unk_token="[UNK]", mask_token="[MASK]")
print("vocab_size :", tokenizer.vocab_size)
print("mask_id    :", tokenizer.mask_token_id, "| pad_id:", tokenizer.pad_token_id)
print("sample tok :", tokenizer.tokenize("Once upon a time there was a little cat.")[:14])

In [ ]:
BLOCK_SIZE = 128
def tok_fn(b):
    return tokenizer(b["text"], add_special_tokens=False)
tt = raw_train.map(tok_fn, batched=True, remove_columns=raw_train.column_names, desc="tok train")
tv = raw_val.map(tok_fn, batched=True, remove_columns=raw_val.column_names, desc="tok val")

def group_texts(b):
    cat = sum(b["input_ids"], [])
    n = (len(cat) // BLOCK_SIZE) * BLOCK_SIZE
    return {"input_ids": [cat[i:i+BLOCK_SIZE] for i in range(0, n, BLOCK_SIZE)]}
lm_train = tt.map(group_texts, batched=True, remove_columns=tt.column_names, desc="group train")
lm_val   = tv.map(group_texts, batched=True, remove_columns=tv.column_names, desc="group val")
print(f"train chunks {len(lm_train):,} | val {len(lm_val):,} | approx {len(lm_train)*BLOCK_SIZE/1e6:.2f}M tokens")

In [ ]:
class DiffusionCollator:
    def __init__(self, tok, eps=0.02, seed=SEED):
        self.mask_id = tok.mask_token_id
        self.eps = eps
        self.gen = torch.Generator().manual_seed(seed)   # Trainer seed 와 분리
    def __call__(self, examples):
        ids = torch.tensor([e["input_ids"] for e in examples], dtype=torch.long)
        B, L = ids.shape
        t = torch.rand(B, generator=self.gen) * (1.0 - self.eps) + self.eps
        mask = torch.rand(B, L, generator=self.gen) < t.unsqueeze(1)
        no = ~mask.any(dim=1)
        if no.any():
            j = torch.randint(0, L, (int(no.sum()),), generator=self.gen)
            mask[no, j] = True
        inp = ids.clone(); inp[mask] = self.mask_id
        lab = ids.clone(); lab[~mask] = -100
        return {"input_ids": inp, "attention_mask": torch.ones(B, L, dtype=torch.long),
                "labels": lab, "t": t}
coll = DiffusionCollator(tokenizer)
print("collator ready, mask_id =", coll.mask_id)

In [ ]:
from transformers import BertConfig, BertForMaskedLM
cfg = BertConfig(vocab_size=tokenizer.vocab_size, hidden_size=256, num_hidden_layers=4,
                 num_attention_heads=4, intermediate_size=1024,
                 max_position_embeddings=BLOCK_SIZE, pad_token_id=tokenizer.pad_token_id)
model = BertForMaskedLM(cfg).to(device)
np_ = model.num_parameters()
emb = tokenizer.vocab_size * cfg.hidden_size
print(f"#params {np_/1e6:.2f}M | embedding share {emb/np_:.1%}  (Ch32: ~70%)")

In [ ]:
from transformers import Trainer, TrainingArguments

class DiffusionTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        t = inputs["t"]; labels = inputs["labels"]
        out = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        B, L, V = out.logits.shape
        per = F.cross_entropy(out.logits.view(-1, V), labels.view(-1),
                              ignore_index=-100, reduction="none").view(B, L)
        loss = ((per.sum(dim=1) / L) / t.to(per.dtype)).mean()
        return (loss, out) if return_outputs else loss

args = TrainingArguments(
    output_dir="./out33", max_steps=30000,
    per_device_train_batch_size=64, per_device_eval_batch_size=64,
    learning_rate=3e-4, weight_decay=0.01, warmup_steps=500,
    lr_scheduler_type="cosine", max_grad_norm=1.0, fp16=USE_FP16,
    logging_steps=250, eval_strategy="steps", eval_steps=2000, save_strategy="no",
    report_to="none", label_names=["labels"], remove_unused_columns=False, seed=SEED)

trainer = DiffusionTrainer(model=model, args=args, train_dataset=lm_train,
                           eval_dataset=lm_val, data_collator=coll)
t0 = time.time(); r = trainer.train(); el = (time.time()-t0)/60
print(f"\n=== summary ===\nelapsed {el:.2f} min | step {r.global_step} | train_loss {r.training_loss:.4f}")
print(f"random baseline ln(V) = {math.log(tokenizer.vocab_size):.4f}")
if torch.cuda.is_available():
    print(f"peak VRAM {torch.cuda.max_memory_allocated()/1024**2:.0f} MiB")

In [ ]:
# Ch 32의 기본 샘플러: 전부 [MASK]에서 confidence 기반 병렬 denoise (MaskGIT식)
@torch.no_grad()
def diffusion_generate(model, length=128, steps=16, temperature=1.0, top_k=50,
                       prompt_ids=None, record_trajectory=False):
    model.eval(); dev=device; mask_id=tokenizer.mask_token_id
    x=torch.full((1,length),mask_id,dtype=torch.long,device=dev)
    fixed=torch.zeros(length,dtype=torch.bool,device=dev)
    if prompt_ids is not None:
        p=torch.tensor(prompt_ids[:length],device=dev); x[0,:len(p)]=p; fixed[:len(p)]=True
    n_gen=int((~fixed).sum().item()); traj=[]
    for step in range(steps):
        logits=model(input_ids=x).logits[0]; probs=logits.softmax(-1)
        if temperature>0:
            scaled=logits/temperature
            if top_k>0:
                kth=scaled.topk(top_k,dim=-1).values[:,-1,None]; scaled=scaled.masked_fill(scaled<kth,float("-inf"))
            pred=torch.multinomial(scaled.softmax(-1),1).squeeze(-1); conf=probs.gather(-1,pred.unsqueeze(-1)).squeeze(-1)
        else:
            conf,pred=probs.max(-1)
        is_mask=(x[0]==mask_id)&(~fixed); x_new=torch.where(is_mask,pred,x[0])
        n_remain=int(round(n_gen*(1.0-(step+1)/steps)))
        if n_remain>0:
            cm=conf.clone(); cm[~is_mask]=float("inf")
            x_new[cm.topk(n_remain,largest=False).indices]=mask_id
        x[0]=x_new
        if record_trajectory: traj.append(x[0].clone())
    text=tokenizer.decode(x[0],skip_special_tokens=True)
    return (text,traj) if record_trajectory else text

pid=tokenizer("Once upon a time",add_special_tokens=False)["input_ids"]
torch.manual_seed(SEED)
print("=== unconditional (전부 [MASK] -> 기본 confidence 샘플러) ===")
for i in range(4): print(f"[{i}] {diffusion_generate(model)[:300]}")
print("\n=== conditional ('Once upon a time') ===")
for i in range(2): print(f"[{i}] {diffusion_generate(model, prompt_ids=pid)[:300]}")

In [ ]:
# 모델 품질 진단: 고정-t(0.15) top-1 acc (샘플러 무관)
g=torch.Generator().manual_seed(0)
def fixed_t_acc(tv=0.15,n=128):
    cor=tot=0
    for ex in lm_val.select(range(min(n,len(lm_val)))):
        ids=torch.tensor(ex["input_ids"]); m=torch.rand(len(ids),generator=g)<tv
        if not m.any(): m[0]=True
        inp=ids.clone(); inp[m]=tokenizer.mask_token_id
        with torch.no_grad(): pr=model(inp.unsqueeze(0).to(device)).logits[0].argmax(-1).cpu()
        cor+=(pr[m]==ids[m]).sum().item(); tot+=int(m.sum())
    return cor/tot
print(f"[diag] fixed-t(0.15) top-1 acc = {fixed_t_acc():.3f}  (모델이 조건부 학습했으면 0.5+)")